## Imports and setup


In [3]:
import os, time, random, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import datasets, transforms, models
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
try:
    import subprocess
    subprocess.run(["nvidia-smi"])
except:
    pass


Using device: cpu


## Dataset and transforms


In [4]:
IMG_SIZE=160
stats=((0.485,0.456,0.406),(0.229,0.224,0.225))

train_transform=transforms.Compose([
    transforms.Resize((IMG_SIZE,IMG_SIZE)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ToTensor(),
    transforms.Normalize(*stats)
])
val_transform=transforms.Compose([
    transforms.Resize((IMG_SIZE,IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(*stats)
])

dataset_dir="dataset"
train_dir=os.path.join(dataset_dir,"train","train")
full=datasets.ImageFolder(train_dir)
class_names=full.classes
total=len(full)
train_n=int(0.6*total); val_n=int(0.2*total); test_n=total-train_n-val_n
train_ds, val_ds, test_ds = random_split(full,[train_n,val_n,test_n],
                                         generator=torch.Generator().manual_seed(SEED))

class SubsetWithTransform(Dataset):
    def __init__(self, subset, transform):
        self.subset=subset
        self.transform=transform
    def __len__(self): return len(self.subset)
    def __getitem__(self, idx):
        x,y=self.subset[idx]
        if self.transform: x=self.transform(x)
        return x,y

train_ds=SubsetWithTransform(train_ds,train_transform)
val_ds=SubsetWithTransform(val_ds,val_transform)
test_ds=SubsetWithTransform(test_ds,val_transform)

## Dataloaders


In [5]:
BATCH_SIZE=64
NUM_WORKERS=2

train_loader=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS)
val_loader=DataLoader(val_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS)
test_loader=DataLoader(test_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS)

print("Classes:", class_names)
print("Samples:", len(train_ds), len(val_ds), len(test_ds))


Classes: ['Apple Braeburn', 'Apple Granny Smith', 'Apricot', 'Avocado', 'Banana', 'Blueberry', 'Cactus fruit', 'Cantaloupe', 'Cherry', 'Clementine', 'Corn', 'Cucumber Ripe', 'Grape Blue', 'Kiwi', 'Lemon', 'Limes', 'Mango', 'Onion White', 'Orange', 'Papaya', 'Passion Fruit', 'Peach', 'Pear', 'Pepper Green', 'Pepper Red', 'Pineapple', 'Plum', 'Pomegranate', 'Potato Red', 'Raspberry', 'Strawberry', 'Tomato', 'Watermelon']
Samples: 10112 3370 3372


## Models


In [6]:
def simple_cnn(num_classes):
    model=nn.Sequential(
        nn.Conv2d(3,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(32,64,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(64,128,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.AdaptiveAvgPool2d(1),
        nn.Flatten(),
        nn.Linear(128,128), nn.ReLU(), nn.Dropout(0.3),
        nn.Linear(128,num_classes)
    )
    return model

def create_resnet18(num_classes, pretrained=True, freeze_backbone=True):
    model=models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
    if freeze_backbone:
        for p in model.parameters(): p.requires_grad=False
    in_feats=model.fc.in_features
    model.fc=nn.Sequential(nn.Dropout(0.3), nn.Linear(in_feats,128),
                           nn.ReLU(), nn.Dropout(0.3),
                           nn.Linear(128,num_classes))
    return model

def create_efficient_b0(num_classes, pretrained=True, freeze_backbone=True):
    model=models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None)
    if freeze_backbone:
        for p in model.parameters(): p.requires_grad=False
    in_feats=model.classifier[1].in_features
    model.classifier=nn.Sequential(nn.Dropout(0.3), nn.Linear(in_feats,128),
                                   nn.ReLU(), nn.Dropout(0.3),
                                   nn.Linear(128,num_classes))
    return model

num_classes=len(class_names)
cnn_model=simple_cnn(num_classes).to(device)
resnet_model=create_resnet18(num_classes).to(device)
eff_model=create_efficient_b0(num_classes).to(device)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\LENOVO/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:08<00:00, 5.35MB/s]



## Training function


In [7]:
def train_model(model,name,train_loader,val_loader,num_epochs=1,lr=1e-3):
    model=model.to(device)
    crit=nn.CrossEntropyLoss()
    opt=optim.Adam(filter(lambda p:p.requires_grad, model.parameters()), lr=lr)
    best_path=f"best_{name}.pth"
    best=0
    hist={'train_loss':[],'val_loss':[],'train_acc':[],'val_acc':[]}
    for ep in range(num_epochs):
        model.train(); tl=0;tc=0;tt=0
        for X,y in train_loader:
            X,y=X.to(device),y.to(device)
            opt.zero_grad()
            out=model(X)
            loss=crit(out,y)
            loss.backward(); opt.step()
            tl+=loss.item()
            preds=out.argmax(1)
            tc+=(preds==y).sum().item(); tt+=y.size(0)
        tr_loss=tl/len(train_loader); tr_acc=tc/tt
        model.eval(); vl=0;vc=0;vt=0
        with torch.no_grad():
            for Xv,yv in val_loader:
                Xv,yv=Xv.to(device),yv.to(device)
                outv=model(Xv)
                lossv=crit(outv,yv)
                vl+=lossv.item()
                pv=outv.argmax(1)
                vc+=(pv==yv).sum().item(); vt+=yv.size(0)
        va_loss=vl/len(val_loader); va_acc=vc/vt
        print(name, ep+1, tr_loss, va_loss, tr_acc, va_acc)
        if va_acc>best:
            best=va_acc
            torch.save(model.state_dict(),best_path)
    return hist,best_path


## Run models


In [ ]:
hist_c, p_c = train_model(cnn_model,"cnn",train_loader,val_loader,num_epochs=3)
hist_r, p_r = train_model(resnet_model,"resnet18",train_loader,val_loader,num_epochs=3)
hist_e, p_e = train_model(eff_model,"effnetb0",train_loader,val_loader,num_epochs=3)


## Evaluation


In [ ]:
def evaluate(model, path, loader):
    model.load_state_dict(torch.load(path,map_location=device))
    model.eval(); preds=[]; labs=[]
    with torch.no_grad():
        for X,y in loader:
            X=X.to(device)
            out=model(X)
            p=out.argmax(1).cpu().numpy()
            preds+=p.tolist(); labs+=y.tolist()
    acc=accuracy_score(labs,preds)
    print("Acc:",acc)
    print(confusion_matrix(labs,preds))
    print(classification_report(labs,preds))
    return acc

print("CNN"); ac_c = evaluate(cnn_model,p_c,test_loader)
print("ResNet18"); ac_r = evaluate(resnet_model,p_r,test_loader)
print("EffNetB0"); ac_e = evaluate(eff_model,p_e,test_loader)
